In [4]:
from datetime import datetime
import pandas as pd
# import ezodf
from pydantic import BaseModel
import requests
import folium
import math
import numpy as np
from typing import List, Optional, Dict, Tuple
from collections import Counter

# ------------------------- Models -------------------------
class Point(BaseModel):
    latitude: float
    longitude: float
    timestamp: float  # Unix timestamp
    yaw_rate: float = 0.0

class LanePosition(BaseModel):
    latitude: float
    longitude: float
    lane_number: int
    confidence: float
    lateral_offset: float
    road_bearing: float
    timestamp: float

class RoadGeometry(BaseModel):
    centerline_lat: float
    centerline_lon: float
    bearing: float
    total_lanes: int = 3
    lane_width: float = 3.5
    curvature: float = 0.0

# ------------------------- Extractor -------------------------
def extract_points_from_csv(file_path: str, lat_col: str = "lat", lon_col: str = "long", 
                           time_col: str = "TimeStamp", yaw_col: str = "yawRate") -> list[Point]:
    df = pd.read_csv(file_path)
    points = []
    base_date = datetime(2025, 2, 21).date()
    
    for _, row in df.iterrows():
        try:
            full_datetime_str = f"{base_date} {row[time_col]}"
            full_datetime = datetime.strptime(full_datetime_str, "%Y-%m-%d %H:%M:%S.%f")
            timestamp = full_datetime.timestamp()
            yawRate = float(row[yaw_col]) if yaw_col in row else 0.0
            lat = float(row[lat_col])
            long = float(row[lon_col])
            point = Point(latitude=lat, longitude=long, timestamp=timestamp, yaw_rate=yawRate)
            points.append(point)
        except (ValueError, TypeError, KeyError):
            continue
    return points

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = phi2 - phi1
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))

# ------------------------- Calculations -------------------------
def calculate_road_bearing(p1: Point, p2: Point) -> float:
    lat1, lon1 = math.radians(p1.latitude), math.radians(p1.longitude)
    lat2, lon2 = math.radians(p2.latitude), math.radians(p2.longitude)
    dlon = lon2 - lon1
    y = math.sin(dlon) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
    return math.atan2(y, x)

def calculate_lateral_offset(gps_point: Point, matched_point: Tuple[float, float], road_bearing: float) -> float:
    # Calculate the perpendicular distance from GPS point to road centerline
    lat_diff = gps_point.latitude - matched_point[0]
    lon_diff = gps_point.longitude - matched_point[1]
    
    # Convert to meters
    lat_meters = lat_diff * 111319.9
    lon_meters = lon_diff * 111319.9 * math.cos(math.radians(gps_point.latitude))
    
    # Project onto perpendicular to road direction
    # Positive offset = right side of road, negative = left side
    lateral_offset = -lat_meters * math.sin(road_bearing) + lon_meters * math.cos(road_bearing)
    
    return lateral_offset

def detect_lane_change(yaw_rates: List[float], threshold: float = 0.08) -> bool:
    """More sensitive lane change detection"""
    if len(yaw_rates) < 3:
        return False
    
    # Check for sustained yaw rate or sudden changes
    avg_yaw_rate = sum(abs(yr) for yr in yaw_rates) / len(yaw_rates)
    max_yaw_rate = max(abs(yr) for yr in yaw_rates)
    
    return avg_yaw_rate > threshold or max_yaw_rate > (threshold * 2)

def assign_lane_probabilistic(lateral_offset: float, road_geometry: RoadGeometry, 
                               yaw_rates: List[float], confidence_factor: float = 1.0, 
                               previous_lane: Optional[int] = None) -> Tuple[int, float]:
    """Fixed lane assignment with better probability distribution"""
    
    total_lanes = road_geometry.total_lanes
    lane_width = road_geometry.lane_width
    
    # Calculate probabilities for each lane
    lane_probabilities = []
    
    for lane_num in range(1, total_lanes + 1):
        # Lane center offset from road centerline
        # Lane 1 = leftmost, Lane 3 = rightmost (for 3-lane road)
        lane_center_offset = (lane_num - (total_lanes + 1) / 2) * lane_width
        
        # Distance from GPS point to lane center
        distance = abs(lateral_offset - lane_center_offset)
        
        # Gaussian probability based on distance
        sigma = lane_width / 3.0  # Standard deviation
        
        # Adjust sigma during lane changes
        if detect_lane_change(yaw_rates):
            sigma *= 1.8  # Increase uncertainty during lane changes
        
        # Calculate probability
        prob = math.exp(-0.5 * (distance / sigma) ** 2)
        
        # Small continuity bonus for previous lane (reduced from 1.3 to 1.1)
        if previous_lane and lane_num == previous_lane:
            prob *= 1.1
            
        lane_probabilities.append(prob)
    
    # Normalize probabilities
    total_prob = sum(lane_probabilities)
    if total_prob == 0:
        # Fallback to center lane if no good match
        return 2, 0.1
    
    lane_probabilities = [p / total_prob for p in lane_probabilities]
    
    # Find best lane
    best_lane_idx = lane_probabilities.index(max(lane_probabilities))
    best_lane = best_lane_idx + 1
    confidence = max(lane_probabilities)
    
    # Debug output for bias checking
    print(f"DEBUG: Lateral offset: {lateral_offset:.3f}m, Lane probs: {[f'{p:.3f}' for p in lane_probabilities]}, Best: {best_lane}")
    
    # Reduce over-reliance on previous lane if confidence is reasonable
    if confidence > 0.5:
        return best_lane, confidence
    
    # Only use previous lane as fallback if confidence is very low
    if confidence < 0.3 and previous_lane:
        return previous_lane, confidence * 0.7
    
    return best_lane, confidence

def calculate_lane_coordinates(road_geometry: RoadGeometry, lane_number: int, dynamic_offset: float = 0.0) -> Tuple[float, float]:
    """Calculate the lat/lon coordinates for a specific lane"""
    # Lane center offset from road centerline
    lane_center_offset = (lane_number - (road_geometry.total_lanes + 1) / 2) * road_geometry.lane_width
    total_offset = lane_center_offset + dynamic_offset
    
    # Perpendicular bearing to road direction
    bearing = road_geometry.bearing + math.pi/2
    
    # Convert offset to lat/lon
    lat_offset = (total_offset * math.cos(bearing)) / 111319.9
    lon_offset = (total_offset * math.sin(bearing)) / (111319.9 * math.cos(math.radians(road_geometry.centerline_lat)))
    
    return road_geometry.centerline_lat + lat_offset, road_geometry.centerline_lon + lon_offset

# ------------------------- Fixed Lane Mapping -------------------------
def process_lane_mapping(points: List[Point], osrm_data: Dict, start_index: int = 0) -> List[Tuple[int, LanePosition]]:
    """
    Completely fixed version that processes all valid points
    """
    lane_positions = []
    tracepoints = osrm_data.get("tracepoints", [])
    
    if len(tracepoints) != len(points):
        print(f"WARNING: Mismatch - {len(points)} GPS points but {len(tracepoints)} tracepoints")
    
    previous_lane = None
    lane_history = []

    for i, point in enumerate(points):
        global_index = start_index + i

        # Skip if this point has the same coordinates as the previous point
        if i > 0:
            prev_point = points[i - 1]
            if (point.latitude == prev_point.latitude) and (point.longitude == prev_point.longitude):
                print(f"DEBUG: GPS {global_index} - Skipped duplicate coordinate")
                continue

        # Skip if no corresponding tracepoint
        if i >= len(tracepoints):
            print(f"DEBUG: GPS {global_index} - No tracepoint available")
            continue
            
        tracepoint = tracepoints[i]
        
        # Skip if tracepoint is None or has no location
        if not tracepoint or not tracepoint.get("location"):
            print(f"DEBUG: GPS {global_index} - Invalid tracepoint: {tracepoint}")
            continue

        matched_lon, matched_lat = tracepoint["location"]
        
        # Calculate road bearing with better logic
        road_bearing = 0.0
        bearing_calculated = False
        
        # Try to get bearing from next point
        if i < len(points) - 1 and i + 1 < len(tracepoints):
            next_tp = tracepoints[i + 1]
            if next_tp and next_tp.get("location"):
                next_lon, next_lat = next_tp["location"]
                # Only calculate if points are sufficiently different
                if haversine_distance(matched_lat, matched_lon, next_lat, next_lon) > 1.0:
                    road_bearing = calculate_road_bearing(
                        Point(latitude=matched_lat, longitude=matched_lon, timestamp=0),
                        Point(latitude=next_lat, longitude=next_lon, timestamp=0)
                    )
                    bearing_calculated = True
        
        # Try to get bearing from previous point if next didn't work
        if not bearing_calculated and i > 0:
            prev_tp = tracepoints[i - 1]
            if prev_tp and prev_tp.get("location"):
                prev_lon, prev_lat = prev_tp["location"]
                if haversine_distance(prev_lat, prev_lon, matched_lat, matched_lon) > 1.0:
                    road_bearing = calculate_road_bearing(
                        Point(latitude=prev_lat, longitude=prev_lon, timestamp=0),
                        Point(latitude=matched_lat, longitude=matched_lon, timestamp=0)
                    )
                    bearing_calculated = True

        # Create road geometry
        road_geometry = RoadGeometry(
            centerline_lat=matched_lat,
            centerline_lon=matched_lon,
            bearing=road_bearing
        )

        # Calculate lateral offset
        lateral_offset = calculate_lateral_offset(point, (matched_lat, matched_lon), road_bearing)
        
        # Get recent yaw rates for lane change detection
        recent_yaw_rates = []
        for j in range(max(0, i - 2), min(len(points), i + 3)):
            recent_yaw_rates.append(points[j].yaw_rate)
        
        # Assign lane with improved algorithm
        lane_number, confidence = assign_lane_probabilistic(
            lateral_offset, road_geometry, recent_yaw_rates, 1.0, previous_lane
        )
        
        # Apply minimal temporal smoothing
        lane_history.append(lane_number)
        if len(lane_history) > 3:  # Reduced from 5 to 3
            lane_history.pop(0)
            
        # Only smooth if we have enough history and confidence is very low
        if len(lane_history) >= 3 and confidence < 0.4:
            lane_counts = Counter(lane_history)
            most_common_lane = lane_counts.most_common(1)[0][0]
            
            if lane_number != most_common_lane:
                print(f"DEBUG: GPS {global_index} - Smoothing lane {lane_number} -> {most_common_lane}")
                lane_number = most_common_lane
                confidence *= 0.9

        # Accept points with reasonable confidence
        if confidence > 0.2:  # Lowered threshold to include more points
            lane_lat, lane_lon = calculate_lane_coordinates(road_geometry, lane_number)

            lane_position = LanePosition(
                latitude=lane_lat,
                longitude=lane_lon,
                lane_number=lane_number,
                confidence=confidence,
                lateral_offset=lateral_offset,
                road_bearing=road_bearing,
                timestamp=point.timestamp
            )
            
            lane_positions.append((global_index, lane_position))
            previous_lane = lane_number
            
            print(f"DEBUG: GPS {global_index} - Mapped to lane {lane_number} (conf: {confidence:.3f}, offset: {lateral_offset:.3f}m)")
        else:
            print(f"DEBUG: GPS {global_index} - Skipped due to low confidence: {confidence:.3f}")

    return lane_positions

# ------------------------- Improved Map Visualization -------------------------
def create_lane_map(points: List[Point], indexed_lane_positions: List[Tuple[int, LanePosition]]) -> folium.Map:
    if not points:
        return folium.Map()
    
    # Center map on first point
    fmap = folium.Map(location=[points[0].latitude, points[0].longitude], zoom_start=17)
    lane_colors = {1: 'red', 2: 'blue', 3: 'green', 4: 'orange', 5: 'purple'}

    # Plot all GNSS points (small purple circles)
    for i, point in enumerate(points):
        folium.CircleMarker(
            [point.latitude, point.longitude],
            radius=2,
            color='purple',
            fill=True,
            fill_color='purple',
            fillOpacity=0.6,
            popup=f"GNSS Point {i}: {point.latitude:.6f}, {point.longitude:.6f}"
        ).add_to(fmap)

    # Plot lane-mapped points
    mapped_count = 0
    lane_distribution = Counter()
    
    for gnss_index, lp in indexed_lane_positions:
        if gnss_index >= len(points):
            continue
            
        original_point = points[gnss_index]
        lane_distribution[lp.lane_number] += 1
        
        # Create lane-mapped point marker
        folium.CircleMarker(
            [lp.latitude, lp.longitude],
            radius=2,
            color=lane_colors.get(lp.lane_number, 'black'),
            fill=True,
            fill_color=lane_colors.get(lp.lane_number, 'black'),
            fillOpacity=0.8,
            popup=(f"GNSS #{gnss_index} -> Lane {lp.lane_number}<br>"
                   f"Original: {original_point.latitude:.6f}, {original_point.longitude:.6f}<br>"
                   f"Offset: {lp.lateral_offset:.2f} m<br>"
                   f"Confidence: {lp.confidence:.2f}<br>"
                   f"Bearing: {math.degrees(lp.road_bearing):.1f}°")
        ).add_to(fmap)
        
        # Draw line connecting GNSS point to lane-mapped point
        folium.PolyLine(
            [[original_point.latitude, original_point.longitude], [lp.latitude, lp.longitude]],
            color=lane_colors.get(lp.lane_number, 'black'),
            weight=1,
            opacity=0.5
        ).add_to(fmap)
        
        mapped_count += 1

    # Add statistics to legend
    total_points = len(points)
    mapping_rate = (mapped_count / total_points * 100) if total_points > 0 else 0
    
    legend_html = f'''
    <div style="position: fixed; 
                top: 10px; right: 10px; width: 200px; height: 180px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:12px; padding: 10px">
    <p><b>Lane Mapping Results</b></p>

    <p><i class="fa fa-circle" style="color:purple"></i> GNSS Points</p>
    <p><i class="fa fa-circle" style="color:red"></i> Lane 1: {lane_distribution[1]}</p>
    <p><i class="fa fa-circle" style="color:blue"></i> Lane 2: {lane_distribution[2]}</p>
    <p><i class="fa fa-circle" style="color:green"></i> Lane 3: {lane_distribution[3]}</p>
    </div>
    '''
    fmap.get_root().html.add_child(folium.Element(legend_html))

    return fmap

# ------------------------- Main Process -------------------------
def process_with_lane_mapping(file_path: str):
    """Main processing function with improved error handling"""
    print("Extracting GPS points from ODS file...")
    points = extract_points_from_csv(file_path)
    print(f"Extracted {len(points)} GPS points")
    
    if len(points) < 2:
        print("ERROR: Need at least 2 GPS points")
        return [], [], None
    
    BATCH_SIZE = 100
    all_lane_positions = []
    
    print("Processing GPS points in batches...")
    
    for start in range(0, len(points), BATCH_SIZE):
        batch = points[start:start + BATCH_SIZE]
        batch_end = start + len(batch)
        
        print(f"Processing batch {start//BATCH_SIZE + 1}: GPS points {start} to {batch_end-1}")
        
        if len(batch) < 2:
            print("Batch too small, skipping...")
            continue
            
        try:
            # Prepare OSRM request
            coord_str = ';'.join([f"{p.longitude},{p.latitude}" for p in batch])
            radius_str = ';'.join(['50'] * len(batch))
            yaw_rates_str = ';'.join([str(point.yaw_rate) for point in batch])

            url = f"http://127.0.0.1:5001/match/v1/driving/{coord_str}?overview=full&geometries=geojson&gaps=ignore&radiuses={radius_str}&yaw_rate={yaw_rates_str}"
            
            # Make request
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                
                if 'tracepoints' in data:
                    print(f"OSRM returned {len(data['tracepoints'])} tracepoints for {len(batch)} GPS points")
                    
                    # Process this batch
                    batch_lane_positions = process_lane_mapping(batch, data, start)
                    all_lane_positions.extend(batch_lane_positions)
                    
                    print(f"Mapped {len(batch_lane_positions)} points in this batch")
                else:
                    print("ERROR: No tracepoints in OSRM response")
            else:
                print(f"ERROR: OSRM request failed with status {response.status_code}")
                
        except Exception as e:
            print(f"ERROR in batch starting at {start}: {e}")
            continue

    print(f"\nTotal processing complete:")
    print(f"- Input GPS points: {len(points)}")
    print(f"- Successfully mapped: {len(all_lane_positions)}")
    print(f"- Mapping rate: {len(all_lane_positions)/len(points)*100:.1f}%")
    
    # Create visualization
    print("Creating map visualization...")
    fmap = create_lane_map(points, all_lane_positions)
    
    return points, all_lane_positions, fmap

# Execute the processing
if __name__ == "__main__":
    file_path = r"harman_clean.csv"
    points, lane_positions, fmap = process_with_lane_mapping(file_path)
    
    # Print final statistics
    if lane_positions:
        lane_dist = Counter([lp[1].lane_number for lp in lane_positions])
        print(f"\nFinal lane distribution:")
        for lane in sorted(lane_dist.keys()):
            print(f"Lane {lane}: {lane_dist[lane]} points ({lane_dist[lane]/len(lane_positions)*100:.1f}%)")
    
    print("\nDone! Check the map visualization.")
    display(fmap)


Extracting GPS points from ODS file...
Extracted 2591 GPS points
Processing GPS points in batches...
Processing batch 1: GPS points 0 to 99
OSRM returned 100 tracepoints for 100 GPS points
DEBUG: Lateral offset: -6.798m, Lane probs: ['1.000', '0.000', '0.000'], Best: 1
DEBUG: GPS 0 - Mapped to lane 1 (conf: 1.000, offset: -6.798m)
DEBUG: GPS 1 - Skipped duplicate coordinate
DEBUG: GPS 2 - Skipped duplicate coordinate
DEBUG: GPS 3 - Skipped duplicate coordinate
DEBUG: GPS 4 - Skipped duplicate coordinate
DEBUG: GPS 5 - Skipped duplicate coordinate
DEBUG: GPS 6 - Skipped duplicate coordinate
DEBUG: GPS 7 - Skipped duplicate coordinate
DEBUG: GPS 8 - Skipped duplicate coordinate
DEBUG: Lateral offset: -6.943m, Lane probs: ['1.000', '0.000', '0.000'], Best: 1
DEBUG: GPS 9 - Mapped to lane 1 (conf: 1.000, offset: -6.943m)
DEBUG: GPS 10 - Skipped duplicate coordinate
DEBUG: GPS 11 - Skipped duplicate coordinate
DEBUG: GPS 12 - Skipped duplicate coordinate
DEBUG: GPS 13 - Skipped duplicate co